In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table= f"{catalog_name}.{bronze_schema}.races"
silver_table= f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df= spark.table(bronze_table)

In [0]:
races_drop_df= races_df.drop("url")

In [0]:
from pyspark.sql import functions as F

In [0]:
races_rename_df = ( races_drop_df
                  .withColumnRenamed("raceName", "race_name")
.withColumnRenamed("circuitId", "circuit_id")
.withColumnRenamed("date","race_date")
.withColumn("batch_id", F.lit(v_batch_id)))

In [0]:
display(races_rename_df)


In [0]:
races_dup_df= races_rename_df.dropDuplicates(["season", "round"])

In [0]:
display(races_dup_df)

In [0]:
races_valid_df =  races_dup_df.withColumn("race_name", F.initcap(F.col("race_name")))

In [0]:
display(races_valid_df)

In [0]:
write_to_silver(
    input_df= races_valid_df,
    target_table= silver_table,
    merge_condition= "t.season= s.season AND t.round= s.round",
    columns_to_update=[
        "circuit_id",
        "race_name",
        "race_date",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))